In [1]:
from pathlib import Path
from supabase import create_client, Client
import mimetypes
from dotenv import load_dotenv
import os

load_dotenv()

SUPABASE_URL = os.getenv("SUPABASE_URL")
SUPABASE_SERVICE_ROLE_KEY = os.getenv("SUPABASE_SERVICE_ROLE_KEY")
BUCKET = "pictures"

In [42]:
# local root folder that contains products/...
LOCAL_ROOT = Path(r"C:\github\kcw-product-pictures\pictures")

# optional: only upload under this subfolder in local root
START_SUBFOLDER = "product"

supabase: Client = create_client(SUPABASE_URL, SUPABASE_SERVICE_ROLE_KEY)


def guess_content_type(path: Path) -> str:
    ctype, _ = mimetypes.guess_type(str(path))
    return ctype or "application/octet-stream"


def iter_files(root: Path):
    for p in root.rglob("*"):
        if p.is_file():
            yield p


def upload_file(local_file: Path, storage_path: str):
    content_type = guess_content_type(local_file)

    with open(local_file, "rb") as f:
        data = f.read()

    return supabase.storage.from_(BUCKET).upload(
        path=storage_path,
        file=data,
        file_options={
            "content-type": content_type,
            "upsert": False,
        },
    )


def main():
    base = LOCAL_ROOT / START_SUBFOLDER if START_SUBFOLDER else LOCAL_ROOT

    if not base.exists():
        raise FileNotFoundError(f"Folder not found: {base}")

    success = 0
    failed = 0

    for local_file in iter_files(base):
        rel_path = local_file.relative_to(LOCAL_ROOT).as_posix()
    
        try:
            upload_file(local_file, rel_path)
            print(f"[UPLOADED] {rel_path}")
            success += 1
    
        except Exception as e:
            msg = str(e).lower()
    
            if "already exists" in msg or "duplicate" in msg:
                print(f"[SKIP] {rel_path}")
            else:
                print(f"[FAIL] {rel_path} -> {e}")
                failed += 1

    print(f"\nDONE | success={success} failed={failed}")


if __name__ == "__main__":
    main()

[SKIP] product/01010019/01010019.jpg
[SKIP] product/01010030/01010030.jpg
[SKIP] product/01010140/01010140.jpg
[SKIP] product/01010174/01010174.jpg
[SKIP] product/01010226/01010226.JPG
[SKIP] product/01010241/01010241.jpg
[SKIP] product/01010242/01010242.jpg
[SKIP] product/01010300/01010300.jpg
[SKIP] product/01010301/01010301.jpg
[SKIP] product/01010450/01010450.jpg
[SKIP] product/01010470/01010470.jpg
[SKIP] product/01010650/01010650.JPG
[SKIP] product/01010671/01010671.jpg
[SKIP] product/01011102/01011102.jpg
[SKIP] product/01011102/01011102_2.jpg
[SKIP] product/01011150/01011150.jpg
[SKIP] product/01011190/01011190.JPG
[SKIP] product/01011191/01011191.JPG
[SKIP] product/01011270/01011270.jpg
[SKIP] product/01011300/01011300.jpg
[SKIP] product/01011360/01011360.jpg
[SKIP] product/01011451/01011451.jpg
[SKIP] product/01011520/01011520.jpg
[SKIP] product/01011630/01011630.jpg
[SKIP] product/01011860/01011860.jpg
[SKIP] product/01011980/01011980.jpg
[SKIP] product/01011980/01011980_2.j

In [36]:
from supabase import create_client
from dotenv import load_dotenv
import os

load_dotenv()

SUPABASE_URL = os.getenv("SUPABASE_URL")
SUPABASE_SERVICE_ROLE_KEY = os.getenv("SUPABASE_SERVICE_ROLE_KEY")

BUCKET = "pictures"

SKIP_PREFIXES = ("knowledge", "product", "public")

supabase = create_client(SUPABASE_URL, SUPABASE_SERVICE_ROLE_KEY)


def list_root():
    """
    list root objects / folders
    """
    return supabase.storage.from_(BUCKET).list(path="")


def list_recursive(prefix):
    """
    recursively list all files under prefix
    """
    stack = [prefix]
    results = []

    while stack:
        current = stack.pop()
        items = supabase.storage.from_(BUCKET).list(path=current)

        for it in items:
            name = it["name"]
            is_folder = it.get("id") is None   # supabase trick

            full = f"{current}/{name}" if current else name

            if is_folder:
                stack.append(full)
            else:
                results.append(full)

    return results


def main():

    root_items = list_root()

    for item in root_items:

        name = item["name"]

        if name in SKIP_PREFIXES:
            print(f"SKIP folder: {name}")
            continue

        print(f"\nScanning folder: {name}")

        files = list_recursive(name)

        for old_path in files:

            new_path = f"product/{old_path}"

            print(f"Move {old_path} -> {new_path}")

            try:
                supabase.storage.from_(BUCKET).move(old_path, new_path)
            except Exception as e:
                print("ERROR:", e)


if __name__ == "__main__":
    # main()

SKIP folder: knowledge
SKIP folder: product
SKIP folder: public

Scanning folder: restructure_images.py
